# Interactive exploration

An **analysis/teaching layer** on top of the library — not the reproducible
experiment path. Pick a **regime** (a preset scenario) *or* tweak the controls
(population mix, information, ecological knowledge, decision noise, communication),
then hit **▶ Run & animate** to watch that exact config unfold: a top-down animation
(central pool + agents around it), the resource/harvest trajectories, and the
headline metrics.

> The ring layout in the animation is **decorative** — our agents have no positions
> and the resource is one shared scalar. It illustrates the (non-spatial) dynamics;
> it is not a spatial simulation.

For reproducible results use the CLI (`emergent-coop run --config ...`) and the
scripts in `scripts/`; the findings are written up in
[`docs/findings-summary.md`](../docs/findings-summary.md).

**Setup:** `pip install -e ".[notebook]"`, then **Run All**.

> ⚠️ **Kernel:** if you get `ModuleNotFoundError: No module named 'emergent_cooperation'`,
> the notebook is running under the wrong Python. Select the interpreter/kernel that is
> this repo's **`.venv`** (in VS Code: *Select Kernel → Python Environments →* the
> `repo\.venv` one; in JupyterLab: the *"emergent-coop (.venv)"* kernel) — not a
> different project's `.venv` or the global Python.

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import HTML, display

import emergent_cooperation
from emergent_cooperation.core.config import AgentSpec, ResourceConfig, SimulationConfig
from emergent_cooperation.core.simulation import run_simulation
from emergent_cooperation.metrics.metrics import compute_metrics

# Reuse the animation builder from scripts/ (locate the repo via the installed package,
# so this works no matter what the notebook's working directory is).
REPO = Path(emergent_cooperation.__file__).resolve().parents[2]
sys.path.insert(0, str(REPO / "scripts"))
from animate_run import make_animation  # noqa: E402

GROUP_SIZE = 8
COOP_TYPES = ["cooperative", "conditional_cooperator", "compensating_cooperator", "sanctioning"]


def build_config(cooperator_type, n_selfish, information_model, knowledge_bias,
                 decision_noise, broadcast_reliability, rounds, seed):
    """Turn the dashboard controls into a SimulationConfig."""
    n_coop = GROUP_SIZE - n_selfish
    params = {"regeneration_rate": 0.4, "capacity": 100.0}
    if cooperator_type == "sanctioning":
        params["monitoring_cost"] = 0.2
    else:  # cooperative / conditional / compensating all accept knowledge_bias
        params["knowledge_bias"] = knowledge_bias
    agents = []
    if n_coop > 0:
        agents.append(AgentSpec(cooperator_type, n_coop, params))
    if n_selfish > 0:
        agents.append(AgentSpec("selfish", n_selfish, {"greed": 1.0}))
    return SimulationConfig(
        name="explore", rounds=rounds, information_model=information_model,
        decision_noise=decision_noise, broadcast_reliability=broadcast_reliability,
        resource=ResourceConfig(initial_level=50.0, capacity=100.0,
                                regeneration_rate=0.4, collapse_threshold=1.0),
        agents=tuple(agents),
    )


def run_and_show(**controls):
    """Run one config and display: the animated ring view, the trajectory, and metrics."""
    cfg = build_config(**controls)
    result = run_simulation(cfg, seed=controls["seed"])
    m = compute_metrics(result, capacity=100.0, regeneration_rate=0.4, collapse_threshold=1.0)
    n_coop = GROUP_SIZE - controls["n_selfish"]
    label = (f"{n_coop} {controls['cooperator_type']} + {controls['n_selfish']} selfish  "
             f"[{controls['information_model']}]")

    # 1) animated top-down view (illustrative: the ring is decorative)
    fig, anim = make_animation(result, label)
    html = anim.to_jshtml()
    plt.close(fig)
    display(HTML(html))

    # 2) the same run as trajectories, plus the headline metrics
    x = [r.round_index for r in result.rounds]
    fig2, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2))
    a1.plot(x, [r.resource_after_harvest for r in result.rounds])
    a1.axhline(50, color="grey", ls=":", lw=1)
    a1.set_ylim(-2, 100); a1.set_xlabel("round"); a1.set_ylabel("stock")
    a1.set_title("resource over time")
    a2.plot(x, [r.total_harvested for r in result.rounds], color="tab:green")
    a2.axhline(10, color="grey", ls=":", lw=1)
    a2.set_xlabel("round"); a2.set_ylabel("harvest"); a2.set_title("harvest over time")
    fig2.tight_layout(); plt.show()

    print(f"sustainability={m['sustainability_ratio']:.2f}   collapsed={m['collapsed']}   "
          f"gini={m['payoff_gini']:.2f}   total_harvest={m['total_harvest']:.0f}")

In [ ]:
# Pick a REGIME (a preset config) or tweak the controls, then "▶ Run & animate".
REGIMES = {
    "— custom (use the controls) —": None,
    "🎣 Tragedy of the commons (all selfish)": dict(
        cooperator_type="cooperative", n_selfish=8, information_model="global",
        knowledge_bias=1.0, decision_noise=0.0, broadcast_reliability=0.0),
    "🌱 Sustainable cooperation": dict(
        cooperator_type="cooperative", n_selfish=0, information_model="global",
        knowledge_bias=1.0, decision_noise=0.0, broadcast_reliability=0.0),
    "🪝 Free-riding erodes it": dict(
        cooperator_type="cooperative", n_selfish=3, information_model="global",
        knowledge_bias=1.0, decision_noise=0.0, broadcast_reliability=0.0),
    "↩️ Reciprocity collapses it (E2)": dict(
        cooperator_type="conditional_cooperator", n_selfish=2, information_model="global",
        knowledge_bias=1.0, decision_noise=0.0, broadcast_reliability=0.0),
    "🛡️ Enforcement holds (E3)": dict(
        cooperator_type="sanctioning", n_selfish=4, information_model="global",
        knowledge_bias=1.0, decision_noise=0.0, broadcast_reliability=0.0),
    "🙈 Blind & overconfident (E1 / H6)": dict(
        cooperator_type="cooperative", n_selfish=0, information_model="private",
        knowledge_bias=1.3, decision_noise=0.0, broadcast_reliability=0.0),
    "📣 Communication → fairness (E6)": dict(
        cooperator_type="conditional_cooperator", n_selfish=2, information_model="private",
        knowledge_bias=1.0, decision_noise=0.0, broadcast_reliability=1.0),
}

w = dict(
    cooperator_type=widgets.Dropdown(
        options=COOP_TYPES, value="cooperative", description="cooperator"),
    n_selfish=widgets.IntSlider(min=0, max=8, value=0, description="# selfish"),
    information_model=widgets.Dropdown(
        options=["global", "private"], value="global", description="info"),
    knowledge_bias=widgets.FloatSlider(
        min=0.6, max=1.5, step=0.1, value=1.0, description="knowledge"),
    decision_noise=widgets.FloatSlider(min=0.0, max=0.5, step=0.05, value=0.0, description="noise"),
    broadcast_reliability=widgets.FloatSlider(
        min=0.0, max=1.0, step=0.1, value=0.0, description="broadcast"),
    rounds=widgets.IntSlider(min=20, max=60, step=10, value=40, description="rounds"),
    seed=widgets.IntSlider(min=1, max=10, value=1, description="seed"),
)
regime = widgets.Dropdown(options=list(REGIMES), description="Regime",
                          layout=widgets.Layout(width="380px"))
run_btn = widgets.Button(description="▶ Run & animate", button_style="success")
out = widgets.Output()
_applying = {"on": False}


def _apply_regime(change):
    preset = REGIMES.get(change["new"])
    if preset is None:
        return
    _applying["on"] = True
    for key, value in preset.items():
        w[key].value = value
    _applying["on"] = False


def _to_custom(change):
    if not _applying["on"] and regime.value != "— custom (use the controls) —":
        regime.unobserve(_apply_regime, "value")
        regime.value = "— custom (use the controls) —"
        regime.observe(_apply_regime, "value")


regime.observe(_apply_regime, "value")
for key in ("cooperator_type", "n_selfish", "information_model",
            "knowledge_bias", "decision_noise", "broadcast_reliability"):
    w[key].observe(_to_custom, "value")


def _on_run(_):
    out.clear_output(wait=True)
    with out:
        run_and_show(**{key: widget.value for key, widget in w.items()})


run_btn.on_click(_on_run)
display(widgets.VBox([
    regime,
    widgets.HBox([w["cooperator_type"], w["n_selfish"]]),
    widgets.HBox([w["information_model"], w["knowledge_bias"]]),
    widgets.HBox([w["decision_noise"], w["broadcast_reliability"]]),
    widgets.HBox([w["rounds"], w["seed"]]),
    run_btn,
]), out)

### The regimes map to the experiments

| Regime | What you'll see | Experiment |
| ------ | --------------- | ---------- |
| 🎣 Tragedy of the commons | pool emptied in one round | E-baseline |
| 🌱 Sustainable cooperation | steady green pool | E-baseline |
| 🪝 Free-riding erodes it | slow decline to collapse | E2 |
| ↩️ Reciprocity collapses it | retaliation ratchet | E2 |
| 🛡️ Enforcement holds | selfish grabs capped, pool steady | E3 |
| 🙈 Blind & overconfident | private + knowledge 1.3 → collapse | E1 / H6 |
| 📣 Communication → fairness | broadcast lets cooperators react | E6 |

After picking a regime, nudge a single control (e.g. raise **# selfish**, or switch
**info** to `private`) to see where each mechanism breaks — the regime label flips to
*"custom"* so you know you've departed from the preset.